In [ ]:
%%capture
using_colab = True

try:
    import google.colab  # type: ignore
    using_colab = True
except ImportError:
    using_colab = False

if using_colab:
    %pip install  "gpjax==0.11.2" optax


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np

# addtional JAX libs
import gpjax as gpx
import optax

# graphics stuff
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# for rendering in vscdoe 
if using_colab:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

In [ ]:
# ------------------------------------------------------------
# Ground-truth latent-space dummy function in n dimensions
# X shape: (N, D)
# returns shape: (N, 1)
# ------------------------------------------------------------
def latent_function(X: jnp.ndarray) -> jnp.ndarray:
    """
    Dummy nD latent-space function.

    Example:
      f(z) = sin(z0) + cos(2 z1) + 0.3 sin(z2 z0) + 0.1 ||z||^2

    Works for any D >= 1.
    """
    z0 = X[:, 0]
    z1 = X[:, 1] if X.shape[1] > 1 else 0.0
    z2 = X[:, 2] if X.shape[1] > 2 else 0.0

    y = (
        jnp.sin(z0)
        + jnp.cos(2.0 * z1)
        + 0.3 * jnp.sin(z2 * z0)
        + 0.05 * jnp.sum(X**2, axis=1)
    )
    return y.reshape(-1, 1)


# ------------------------------------------------------------
# Fit GP model with ARD kernel
# ------------------------------------------------------------
def fit_gp(X: jnp.ndarray, y: jnp.ndarray, key: jax.Array):
    latent_dim = X.shape[1]

    kernel = gpx.kernels.Matern52(
        lengthscale=jnp.ones((latent_dim,), dtype=jnp.float64)
    )
    meanf = gpx.mean_functions.Zero()

    prior = gpx.gps.Prior(mean_function=meanf, kernel=kernel)
    likelihood = gpx.likelihoods.Gaussian(num_datapoints=X.shape[0])
    posterior = prior * likelihood

    D = gpx.Dataset(X=X, y=y)

    objective = lambda model, data: -gpx.objectives.conjugate_mll(model, data)
    opt = optax.adam(1e-2)

    posterior, history = gpx.fit(
        model=posterior,
        objective=objective,
        train_data=D,
        optim=opt,
        num_iters=500,
        key=key,
    )

    return posterior, D, history


# ------------------------------------------------------------
# Predictive mean and variance
# ------------------------------------------------------------
def _maybe_call(x):
    return x() if callable(x) else x


def predict_gp(posterior, D, Xtest: jnp.ndarray):
    latent_dist = posterior.predict(Xtest, train_data=D)
    predictive_dist = posterior.likelihood(latent_dist)

    mu = _maybe_call(predictive_dist.mean).reshape(-1)
    var = _maybe_call(predictive_dist.variance).reshape(-1)
    return mu, var


# ------------------------------------------------------------
# Acquisition: uncertainty sampling in nD
# z_next = argmax variance(z), excluding near-duplicates
# ------------------------------------------------------------
def select_next_by_uncertainty(
    posterior,
    D,
    Xcand: jnp.ndarray,
    Xtrain: jnp.ndarray,
    min_dist: float = 1e-3,
):
    _, var = predict_gp(posterior, D, Xcand)

    # Euclidean distance from each candidate to nearest training point
    # Xcand:  (Nc, D)
    # Xtrain: (Nt, D)
    diff = Xcand[:, None, :] - Xtrain[None, :, :]
    dists = jnp.linalg.norm(diff, axis=-1)   # (Nc, Nt)
    min_dists = jnp.min(dists, axis=1)       # (Nc,)

    masked_var = jnp.where(min_dists < min_dist, -jnp.inf, var)

    idx = jnp.argmax(masked_var)
    return Xcand[idx:idx+1], var, idx


# ------------------------------------------------------------
# Sequential uncertainty reduction in nD latent space
# ------------------------------------------------------------
def run_uncertainty_reduction(
    latent_dim: int,
    n_init: int = 8,
    n_iter: int = 8,
    n_candidates: int = 500,
    z_min: float = -3.0,
    z_max: float = 3.0,
    seed: int = 0,
):
    key = jax.random.PRNGKey(seed)

    # Initial training data in nD latent space
    key, subkey = jax.random.split(key)
    X = jax.random.uniform(
        subkey,
        shape=(n_init, latent_dim),
        minval=z_min,
        maxval=z_max,
        dtype=jnp.float64,
    )
    y = latent_function(X)

    # Fixed candidate pool in nD latent space
    key, subkey = jax.random.split(key)
    Xcand = jax.random.uniform(
        subkey,
        shape=(n_candidates, latent_dim),
        minval=z_min,
        maxval=z_max,
        dtype=jnp.float64,
    )

    history = []

    for i in range(n_iter):
        key, subkey = jax.random.split(key)

        # GP on current dataset
        posterior, D, _ = fit_gp(X, y, subkey)
        mu, var = predict_gp(posterior, D, Xcand)

        # Select next point from current uncertainty
        Xnext, cand_var, idx = select_next_by_uncertainty(
            posterior=posterior,
            D=D,
            Xcand=Xcand,
            Xtrain=X,
        )

        ynext = latent_function(Xnext)

        # Store acquisition state BEFORE augmentation
        history.append(
            {
                "iter": i + 1,
                "stage": "acquisition",
                "X": X,
                "y": y,
                "Xcand": Xcand,
                "mu": mu,
                "var": var,
                "Xnext": Xnext,
                "ynext": ynext,
                "idx_next": idx,
            }
        )

        print(
            f"iter {i+1:02d}: "
            f"z_next = {jnp.asarray(Xnext[0])}, "
            f"std = {float(jnp.sqrt(cand_var[idx])):.4f}, "
            f"y_next = {float(ynext[0,0]):.4f}"
        )

        # Augment dataset
        X = jnp.concatenate([X, Xnext], axis=0)
        y = jnp.concatenate([y, ynext], axis=0)

    # ------------------------------------------------------------
    # Final GP fit after last acquired point has been added
    # ------------------------------------------------------------
    key, subkey = jax.random.split(key)
    posterior_final, D_final, _ = fit_gp(X, y, subkey)
    mu_final, var_final = predict_gp(posterior_final, D_final, Xcand)

    history.append(
        {
            "iter": n_iter,
            "stage": "final",
            "X": X,
            "y": y,
            "Xcand": Xcand,
            "mu": mu_final,
            "var": var_final,
            "Xnext": X[-1:],
            "ynext": y[-1:],
            "idx_next": None,
            "posterior": posterior_final,
            "D": D_final,
        }
    )

    return X, y, Xcand, history

In [ ]:
X, y, Xcand, history = run_uncertainty_reduction(
    latent_dim=2,
    n_init=10,
    n_iter=12,
    n_candidates=1000,
    z_min=-3.0,
    z_max=3.0,
    seed=42,
)

In [ ]:
def _gaussian_grid(
    xy,
    values,
    nx=60,
    ny=60,
    pad=0.05,
    sigma=None,
    eps=1e-12,
):
    """
    Interpolate scattered (x, y, value) data to a regular grid
    using Gaussian kernel smoothing.

    This is usually much smoother and less spiky than sharp IDW.
    """
    xy = np.asarray(xy, dtype=float)
    values = np.asarray(values, dtype=float).reshape(-1)

    x = xy[:, 0]
    y = xy[:, 1]

    dx = max(np.ptp(x), eps)
    dy = max(np.ptp(y), eps)

    xmin, xmax = x.min() - pad * dx, x.max() + pad * dx
    ymin, ymax = y.min() - pad * dy, y.max() + pad * dy

    xi = np.linspace(xmin, xmax, nx)
    yi = np.linspace(ymin, ymax, ny)
    XI, YI = np.meshgrid(xi, yi)

    if sigma is None:
        sigma = 0.04 * max(dx, dy)

    dist2 = (XI[..., None] - x[None, None, :]) ** 2 + (YI[..., None] - y[None, None, :]) ** 2
    w = np.exp(-0.5 * dist2 / max(sigma**2, eps))

    ZI = np.sum(w * values[None, None, :], axis=-1) / np.maximum(np.sum(w, axis=-1), eps)
    return XI, YI, ZI


def _idw_grid(xy, values, nx=60, ny=60, pad=0.05, power=6., eps=1e-12):
    """
    Interpolate scattered (x, y, value) data to a regular grid using IDW.
    """
    xy = np.asarray(xy, dtype=float)
    values = np.asarray(values, dtype=float).reshape(-1)

    x = xy[:, 0]
    y = xy[:, 1]

    dx = max(np.ptp(x), eps)
    dy = max(np.ptp(y), eps)

    xmin, xmax = x.min() - pad * dx, x.max() + pad * dx
    ymin, ymax = y.min() - pad * dy, y.max() + pad * dy

    xi = np.linspace(xmin, xmax, nx)
    yi = np.linspace(ymin, ymax, ny)
    XI, YI = np.meshgrid(xi, yi)

    dist2 = (XI[..., None] - x[None, None, :]) ** 2 + (YI[..., None] - y[None, None, :]) ** 2
    w = 1.0 / np.maximum(dist2, eps) ** (power / 2.0)

    ZI = np.sum(w * values[None, None, :], axis=-1) / np.sum(w, axis=-1)
    return XI, YI, ZI

def _nearest_values(Xcand, Xquery, values):
    """
    Assign a z-value to query points by nearest-neighbor lookup in candidate pool.
    """
    Xcand = np.asarray(Xcand, dtype=float)
    Xquery = np.asarray(Xquery, dtype=float)
    values = np.asarray(values, dtype=float).reshape(-1)

    d = np.linalg.norm(Xcand[:, None, :] - Xquery[None, :, :], axis=-1)
    idx = np.argmin(d, axis=0)
    return values[idx]


def make_uncertainty_animation_surface_2d(
    Xcand,
    history,
    title="Sequential uncertainty reduction of dummy latent space function",
    grid_res=70,
    sigma=None,
):
    """
    2D latent-space animation:
      - left: GP mean surface
      - right: GP std surface

    Assumes Xcand is 2D latent data or uses first two columns.
    """
    Xcand = np.asarray(Xcand, dtype=float)
    if Xcand.shape[1] < 2:
        raise ValueError(f"Expected Xcand with at least 2 latent dimensions, got shape {Xcand.shape}.")

    XYcand = Xcand[:, :2]

    all_mu = []
    all_std = []
    for h in history:
        all_mu.append(np.asarray(h["mu"], dtype=float).reshape(-1))
        all_std.append(np.sqrt(np.maximum(np.asarray(h["var"], dtype=float).reshape(-1), 0.0)))

    mu_min = min(v.min() for v in all_mu)
    mu_max = max(v.max() for v in all_mu)
    std_min = min(v.min() for v in all_std)
    std_max = max(v.max() for v in all_std)

    # fixed x/y ranges so animation does not autoscale
    dx = max(np.ptp(XYcand[:, 0]), 1e-12)
    dy = max(np.ptp(XYcand[:, 1]), 1e-12)
    x_range = [XYcand[:, 0].min() - 0.05 * dx, XYcand[:, 0].max() + 0.05 * dx]
    y_range = [XYcand[:, 1].min() - 0.05 * dy, XYcand[:, 1].max() + 0.05 * dy]

    z_range_mu = [mu_min, mu_max]
    z_range_std = [std_min, std_max]

    frames = []
    slider_steps = []

    for k, h in enumerate(history):
        Xk = np.asarray(h["X"], dtype=float)
        yk = np.asarray(h["y"], dtype=float).reshape(-1)
        XYk = Xk[:, :2]

        mu = np.asarray(h["mu"], dtype=float).reshape(-1)
        var = np.asarray(h["var"], dtype=float).reshape(-1)
        std = np.sqrt(np.maximum(var, 0.0))

        stage = h.get("stage", "acquisition")
        iter_label = h.get("iter", k + 1)

        # smoother surfaces
        XI_mu, YI_mu, ZI_mu = _gaussian_grid(XYcand, mu, nx=grid_res, ny=grid_res, sigma=sigma)
        XI_std, YI_std, ZI_std = _gaussian_grid(XYcand, std, nx=grid_res, ny=grid_res, sigma=sigma)

        z_samples_mu = _nearest_values(XYcand, XYk, mu)
        z_samples_std = _nearest_values(XYcand, XYk, std)

        if stage == "acquisition":
            Xnext = np.asarray(h["Xnext"], dtype=float)
            XYnext = Xnext[:, :2]
            ynext = float(np.asarray(h["ynext"], dtype=float).reshape(-1)[0])

            z_next_mu = _nearest_values(XYcand, XYnext, mu)
            z_next_std = _nearest_values(XYcand, XYnext, std)

            next_trace_mu = go.Scatter3d(
                x=XYnext[:, 0],
                y=XYnext[:, 1],
                z=z_next_mu,
                mode="markers",
                name="new candidate",
                marker=dict(size=7, color="red", symbol="diamond"),
                hovertemplate=f"new y={ynext:.3f}<extra></extra>",
            )
            next_trace_std = go.Scatter3d(
                x=XYnext[:, 0],
                y=XYnext[:, 1],
                z=z_next_std,
                mode="markers",
                name="new candidate",
                marker=dict(size=7, color="red", symbol="diamond"),
                hovertemplate=f"new y={ynext:.3f}<extra></extra>",
            )
            subtitle = f"Iteration {iter_label} — acquisition"
        else:
            XYlast = XYk[-1:]
            ylast = float(yk[-1])

            z_last_mu = _nearest_values(XYcand, XYlast, mu)
            z_last_std = _nearest_values(XYcand, XYlast, std)

            next_trace_mu = go.Scatter3d(
                x=XYlast[:, 0],
                y=XYlast[:, 1],
                z=z_last_mu,
                mode="markers",
                name="last added point",
                marker=dict(size=7, color="green", symbol="circle"),
                hovertemplate=f"last y={ylast:.3f}<extra></extra>",
            )
            next_trace_std = go.Scatter3d(
                x=XYlast[:, 0],
                y=XYlast[:, 1],
                z=z_last_std,
                mode="markers",
                name="last added point",
                marker=dict(size=7, color="green", symbol="circle"),
                hovertemplate=f"last y={ylast:.3f}<extra></extra>",
            )
            subtitle = f"Iteration {iter_label} — final refit"

        frame = go.Frame(
            name=str(k),
            data=[
                go.Surface(
                    x=XI_mu,
                    y=YI_mu,
                    z=ZI_mu,
                    colorscale="Plasma",
                    cmin=mu_min,
                    cmax=mu_max,
                    showscale=True,
                    colorbar=dict(title="GP mean"),
                    name="GP mean surface",
                    opacity=0.95,
                ),
                go.Scatter3d(
                    x=XYk[:, 0],
                    y=XYk[:, 1],
                    z=z_samples_mu,
                    mode="markers",
                    name="sampled points",
                    marker=dict(size=4, color="black"),
                    text=[f"y={yy:.3f}" for yy in yk],
                    hovertemplate="%{text}<extra></extra>",
                ),
                next_trace_mu,
                go.Surface(
                    x=XI_std,
                    y=YI_std,
                    z=ZI_std,
                    colorscale="Plasma",
                    cmin=std_min,
                    cmax=std_max,
                    showscale=True,
                    colorbar=dict(title="GP std"),
                    name="GP std surface",
                    opacity=0.95,
                ),
                go.Scatter3d(
                    x=XYk[:, 0],
                    y=XYk[:, 1],
                    z=z_samples_std,
                    mode="markers",
                    name="sampled points",
                    marker=dict(size=4, color="black"),
                    showlegend=False,
                    text=[f"y={yy:.3f}" for yy in yk],
                    hovertemplate="%{text}<extra></extra>",
                ),
                next_trace_std,
            ],
            layout=go.Layout(
                title=f"{title}<br><sup>{subtitle}</sup>"
            ),
        )
        frames.append(frame)

        slider_steps.append(
            {
                "args": [
                    [str(k)],
                    {
                        "frame": {"duration": 500, "redraw": True},
                        "mode": "immediate",
                        "transition": {"duration": 300},
                    },
                ],
                "label": subtitle,
                "method": "animate",
            }
        )

    first = history[0]
    X0 = np.asarray(first["X"], dtype=float)
    y0 = np.asarray(first["y"], dtype=float).reshape(-1)
    XY0 = X0[:, :2]

    mu0 = np.asarray(first["mu"], dtype=float).reshape(-1)
    std0 = np.sqrt(np.maximum(np.asarray(first["var"], dtype=float).reshape(-1), 0.0))

    XI_mu0, YI_mu0, ZI_mu0 = _gaussian_grid(XYcand, mu0, nx=grid_res, ny=grid_res, sigma=sigma)
    XI_std0, YI_std0, ZI_std0 = _gaussian_grid(XYcand, std0, nx=grid_res, ny=grid_res, sigma=sigma)

    z_samples_mu0 = _nearest_values(XYcand, XY0, mu0)
    z_samples_std0 = _nearest_values(XYcand, XY0, std0)

    first_stage = first.get("stage", "acquisition")
    first_iter = first.get("iter", 1)

    if first_stage == "acquisition":
        XYmark0 = np.asarray(first["Xnext"], dtype=float)[:, :2]
        ymark0 = float(np.asarray(first["ynext"], dtype=float).reshape(-1)[0])
        z_mark_mu0 = _nearest_values(XYcand, XYmark0, mu0)
        z_mark_std0 = _nearest_values(XYcand, XYmark0, std0)
        mark_name = "new candidate"
        mark_color = "red"
        mark_symbol = "diamond"
        subtitle0 = f"Iteration {first_iter} — acquisition"
    else:
        XYmark0 = XY0[-1:]
        ymark0 = float(y0[-1])
        z_mark_mu0 = _nearest_values(XYcand, XYmark0, mu0)
        z_mark_std0 = _nearest_values(XYcand, XYmark0, std0)
        mark_name = "last added point"
        mark_color = "green"
        mark_symbol = "circle"
        subtitle0 = f"Iteration {first_iter} — final refit"

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=("GP mean surface", "GP std surface"),
        horizontal_spacing=0.04,
    )

    fig.add_trace(
        go.Surface(
            x=XI_mu0, y=YI_mu0, z=ZI_mu0,
            colorscale="Plasma",
            cmin=mu_min, cmax=mu_max,
            showscale=True,
            colorbar=dict(title="GP mean", x=0.46),
            opacity=0.95,
            name="GP mean surface",
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            x=XY0[:, 0], y=XY0[:, 1], z=z_samples_mu0,
            mode="markers",
            name="sampled points",
            marker=dict(size=4, color="black"),
            text=[f"y={yy:.3f}" for yy in y0],
            hovertemplate="%{text}<extra></extra>",
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter3d(
            x=XYmark0[:, 0], y=XYmark0[:, 1], z=z_mark_mu0,
            mode="markers",
            name=mark_name,
            marker=dict(size=7, color=mark_color, symbol=mark_symbol),
            hovertemplate=f"y={ymark0:.3f}<extra></extra>",
        ),
        row=1, col=1,
    )

    fig.add_trace(
        go.Surface(
            x=XI_std0, y=YI_std0, z=ZI_std0,
            colorscale="Plasma",
            cmin=std_min, cmax=std_max,
            showscale=True,
            colorbar=dict(title="GP std", x=1.02),
            opacity=0.95,
            name="GP std surface",
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Scatter3d(
            x=XY0[:, 0], y=XY0[:, 1], z=z_samples_std0,
            mode="markers",
            name="sampled points",
            marker=dict(size=4, color="black"),
            showlegend=False,
            text=[f"y={yy:.3f}" for yy in y0],
            hovertemplate="%{text}<extra></extra>",
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Scatter3d(
            x=XYmark0[:, 0], y=XYmark0[:, 1], z=z_mark_std0,
            mode="markers",
            name=mark_name,
            marker=dict(size=7, color=mark_color, symbol=mark_symbol),
            showlegend=False,
            hovertemplate=f"y={ymark0:.3f}<extra></extra>",
        ),
        row=1, col=2,
    )

    fig.frames = frames

    camera = dict(eye=dict(x=1.45, y=1.45, z=0.85))

    fig.update_layout(
        title=f"{title}<br><sup>{subtitle0}</sup>",
        template="plotly_white",
        sliders=[
            {
                "active": 0,
                "pad": {"t": 40},
                "steps": slider_steps,
                "currentvalue": {"prefix": "Frame: "},
            }
        ],
        updatemenus=[
            {
                "type": "buttons",
                "showactive": False,
                "x": 1.02,
                "y": 1.15,
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "frame": {"duration": 700, "redraw": True},
                                "fromcurrent": True,
                                "transition": {"duration": 250},
                            },
                        ],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "frame": {"duration": 0, "redraw": False},
                                "mode": "immediate",
                                "transition": {"duration": 0},
                            },
                        ],
                    },
                ],
            }
        ],
        scene=dict(
            xaxis=dict(title="latent dim 1", range=x_range, autorange=False),
            yaxis=dict(title="latent dim 2", range=y_range, autorange=False),
            zaxis=dict(title="GP mean", range=z_range_mu, autorange=False),
            camera=camera,
        ),
        scene2=dict(
            xaxis=dict(title="latent dim 1", range=x_range, autorange=False),
            yaxis=dict(title="latent dim 2", range=y_range, autorange=False),
            zaxis=dict(title="GP std", range=z_range_std, autorange=False),
            camera=camera,
        ),
    )

    return fig

In [ ]:
fig = make_uncertainty_animation_surface_2d(Xcand, history)
fig.show()